In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # because you're in synchain-absa-emotion/notebooks
sys.path.insert(0, str(repo_root))

In [ ]:
from transformers import Trainer, TrainingArguments, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json
from datetime import datetime
from pathlib import Path
import os
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
from scripts.annotation.parsing import (
    extract_aspects,
    extract_emotion,
    extract_sentiment,
)

In [ ]:
from scripts.modeling.inference import load_trained_model, run_aspect_extraction, run_syntactic_parsing, run_opinion_extraction, run_sentiment_analysis, run_emotion_detection

In [ ]:
from scripts.qwen_model.qwen_model import generate_batch_with_checkpoint, load_model

In [ ]:
from scripts.qwen_model.prompts import (
    prompt_aspect_extraction,
    prompt_emotion_classification,
    prompt_opinion_extraction,
    prompt_sentiment_classification,
    prompt_syntactic_parsing,
)

In [ ]:
example_data = {
   "The more you know \uf8ff\u00fc\u00e5\u00e0 #coronavirusec #coronavirus #washyourhands": {
      "aspects": {
         "coronavirusec": 0,
         "coronavirus": 1,
         "washyourhands": 2
      },
      "aspect_sentiments_raw": {
         "0": "The sentiment towards coronavirusec in the given sentence is neutral. Because the sentence fragment does not provide explicit positive or negative language regarding 'coronavirusec'. The use of multiple hashtags including '#coronavirusec' and '#coronavirus' along with '#washyourhands' suggests engagement in a topic related to coronavirus, potentially for awareness or information sharing, but does not convey a clear positive or negative sentiment towards 'coronavirusec' specifically.",
         "1": "The sentiment towards coronavirus in the given sentence is neutral. Because while the sentence includes terms related to the coronavirus and preventive actions, it does not explicitly express a positive or negative emotion towards the virus itself; rather, it emphasizes awareness and preventive actions.",
         "2": "The sentiment towards washyourhands in the given sentence is positive. Because the use of the hashtag \"#washyourhands\" alongside the phrase \"The more you know\" suggests an endorsement of handwashing as a beneficial practice, particularly in the context of preventing the spread of the coronavirus. This indicates a positive sentiment towards the act of washing hands."
      },
      "aspect_sentiments_label": {
         "0": "neutral",
         "1": "neutral",
         "2": "positive"
      },
      "aspect_syntactic": {
         "0": "Based on the provided CoNLL-U format, the word \"coronavirusec\" (ID: 7) is identified as a noun (NOUN) with the lemma \"coronavirusec\". It has a dependency relationship labeled as \"dobj\" (direct object) with the verb \"know\" (ID: 4). This indicates that \"coronavirusec\" is the direct object of the verb \"know,\" meaning the sentence suggests that the subject (\"you\") knows about \"coronavirusec.\"\n\nAdditionally, \"coronavirusec\" is preceded by \"\uf8ff\u00fc\u00e5\u00e0\" (ID: 5) which modifies it (nmod dependency), and it is marked by the symbol \"#\" (ID: 6), indicating it's part of a hashtag. The structure implies that \"coronavirusec\" is a specific term or concept being referred to within the context of the sentence.",
         "1": "Based on the provided CoNLL-U format, the word \"coronavirus\" appears twice in the sentence, but with different roles:\n\n1. **First Appearance**: \n   - **Word**: \"coronavirusec\"\n   - **ID**: 7\n   - **HEAD**: 4 (which corresponds to \"know\")\n   - **DEPREL**: dobj (direct object)\n   - This suggests that \"coronavirusec\" is the direct object of the verb \"know\". It means the sentence is stating that the speaker knows about \"coronavirusec\".\n\n2. **Second Appearance**:\n   - **Word**: \"coronavirus\"\n   - **ID**: 9\n   - **HEAD**: 11 (which corresponds to \"washyourhands\")\n   - **DEPREL**: compound\n   - This indicates that \"coronavirus\" is part of the compound term \"washyourhands\", which is likely",
         "2": "In the given CoNLL-U format, the word \"washyourhands\" is represented as follows:\n\n- **ID**: 11\n- **TEXT**: washyourhands\n- **LEMMA**: washyourhand\n- **POS**: NOUN\n- **TAG**: NNS (plural noun)\n- **HEAD**: 0 (indicating it's the root of the sentence)\n- **DEPREL**: ROOT (it is the main element of the sentence)\n\nBased on this information, \"washyourhands\" is treated as a plural noun (NNS) and serves as the root of the sentence. This suggests that the sentence is centered around the concept of washing hands, likely emphasizing its importance or relevance in the context of the other elements in the sentence, such as the mentions of coronavirus."
      },
      "aspect_opinions": {
         "0": "Based on the given sentence, it's not explicitly clear what the speaker's opinion is towards 'coronavirusec' since the sentence fragment does not provide enough context to infer sentiment. However, the use of hashtags (#coronavirusec, #coronavirus, #washyourhands) suggests the speaker is engaging in a discussion about coronavirus-related topics and possibly promoting awareness or preventive measures. The inclusion of 'coronavirusec' alongside other relevant hashtags indicates it might be a term used in the context of coronavirus discussions, but without further context, we cannot determine",
         "1": "Based on the given sentence and analysis, the speaker seems to be emphasizing awareness and preventive actions related to the coronavirus. The first appearance of \"coronavirusec\" as a direct object suggests the speaker values knowledge about it. The second mention, as part of the hashtag \"#coronavirus,\" alongside \"#washyourhands,\" indicates a focus on hygiene practices to combat the virus. Overall, the speaker appears to advocate for informed awareness and proactive measures like handwashing to deal with the coronavirus.",
         "2": "Based on the sentence \"The more you know \uf8ff\u00fc\u00e5\u00e0 #coronavirusec #coronavirus #washyourhands,\" the speaker appears to be promoting or endorsing the act of washing hands, especially in the context of combating or being informed about the coronavirus. The use of the hashtag \"#washyourhands\" suggests a positive stance, advocating for hygiene practices as a preventive measure against the spread of the virus. The phrase \"The more you know\" implies an educational or informative tone, reinforcing the idea that awareness and action (like handwashing) are important."
      },
      "aspect_emotions_raw": {
         "0": "Emotion: neutral\n\nReasoning: The sentence provided does not contain explicit emotional language or context that would indicate a particular emotion towards 'coronavirusec'. The use of hashtags suggests engagement with the topic but does not convey a clear emotional stance. Without additional context, the most appropriate label for the emotion expressed towards 'coronavirusec' is neutral.",
         "1": "Emotion: neutral\n\nReasoning: The sentence does not express any clear emotional sentiment towards 'coronavirus'. Instead, it emphasizes awareness and preventive actions, which are factual statements rather than emotional expressions. The use of hashtags and the phrase \"The more you know\" suggest an informative tone rather than an emotional one. There's no indication of fear, anger, hope, or any other strong emotion typically associated with discussions about the coronavirus. Therefore, the most appropriate label is 'neutral'.",
         "2": "Emotion: hopeful\n\nReasoning: The sentence conveys a sense of optimism and encouragement towards taking preventive measures like washing hands to combat the spread of the coronavirus. The phrase \"The more you know\" suggests an empowering and informative tone, which aligns with hopefulness about the situation. The speaker seems to believe that by spreading knowledge and encouraging good hygiene practices, there's a positive outcome to be expected, hence the emotion is hopeful rather than any negative or neutral emotion listed."
      },
      "aspect_emotions_label": {
         "0": "neutral",
         "1": "neutral",
         "2": "hopeful"
      },
      "conllu_parse": "# text = The more you know \uf8ff\u00fc\u00e5\u00e0 #coronavirusec #coronavirus #washyourhands\n1\tThe\tthe\tPRON\tDT\t_\t2\tadvmod\t_\t_\n2\tmore\tmore\tADV\tRBR\t_\t4\tadvmod\t_\t_\n3\tyou\tyou\tPRON\tPRP\t_\t4\tnsubj\t_\t_\n4\tknow\tknow\tVERB\tVBP\t_\t11\tparataxis\t_\t_\n5\t\uf8ff\u00fc\u00e5\u00e0\t\uf8ff\u00fc\u00e5\u00e0\tNOUN\tNN\t_\t7\tnmod\t_\t_\n6\t#\t#\tSYM\t$\t_\t7\tpunct\t_\t_\n7\tcoronavirusec\tcoronavirusec\tNOUN\tNN\t_\t4\tdobj\t_\t_\n8\t#\t#\tSYM\t$\t_\t9\tcompound\t_\t_\n9\tcoronavirus\tcoronavirus\tPROPN\tNNP\t_\t10\tcompound\t_\t_\n10\t#\t#\tSYM\t$\t_\t4\tdep\t_\t_\n11\twashyourhands\twashyourhand\tNOUN\tNNS\t_\t0\tROOT\t_\t_"
   },
   "Tune in to @CNBCWEX around 5:30 a.m. ET: Chris Meekins will be on to discuss the latest updates on the #coronavirus.": {
      "aspects": {
         "coronavirus": 0
      },
      "aspect_sentiments_raw": {
         "0": "The sentiment towards coronavirus in the given sentence is neutral. Because the sentence is purely informational, aimed at notifying viewers about an upcoming discussion on the latest updates regarding the coronavirus without expressing any positive or negative feelings towards it."
      },
      "aspect_sentiments_label": {
         "0": "neutral"
      },
      "aspect_syntactic": {
         "0": "Based on the provided CoNLL-U format, the word \"coronavirus\" (ID: 23) has the following syntactic dependencies:\n\n- **HEAD**: 20 (on)\n- **DEPREL**: pobj (prepositional object)\n\nThis indicates that \"coronavirus\" is the object of the preposition \"on\" (ID: 20). In the context of the sentence, \"coronavirus\" is part of the phrase \"to discuss the latest updates on the coronavirus,\" where it is the topic being discussed.\n\nThe full phrase \"the latest updates on the coronavirus\" (words 17-23) functions as the direct object of the verb \"discuss\" (ID: 16). Therefore, \"coronavirus\" is the subject matter that Chris Meekins will be discussing during his appearance on CNBCWEX at 5:30 a.m. ET."
      },
      "aspect_opinions": {
         "0": "Based solely on the given sentence, the speaker does not express a specific opinion towards the \"coronavirus.\" The sentence is informative, aiming to notify viewers about an upcoming discussion on the latest updates regarding the coronavirus. It does not convey any positive or negative sentiment toward the virus itself. The focus is on providing information about a scheduled broadcast segment."
      },
      "aspect_emotions_raw": {
         "0": "Emotion: neutral\n\nThe sentence provided is purely informational and does not contain any emotional language or context that would suggest the speaker feels optimistic, anxious, scared, or any other emotion listed. The speaker is simply informing the audience about an upcoming segment on CNBCWEX where Chris Meekins will discuss the latest updates on the coronavirus. There's no indication of personal feelings or attitudes toward the coronavirus in the statement, making the emotion neutral."
      },
      "aspect_emotions_label": {
         "0": "neutral"
      },
      "conllu_parse": "# text = Tune in to @CNBCWEX around 5:30 a.m. ET: Chris Meekins will be on to discuss the latest updates on the #coronavirus.\n1\tTune\ttune\tNOUN\tNN\t_\t13\tadvcl\t_\t_\n2\tin\tin\tADP\tIN\t_\t1\tadvmod\t_\t_\n3\tto\tto\tADP\tIN\t_\t1\tprep\t_\t_\n4\t@CNBCWEX\t@CNBCWEX\tPROPN\tNNP\t_\t3\tpobj\t_\t_\n5\taround\taround\tADP\tIN\t_\t1\tprep\t_\t_\n6\t5:30\t5:30\tNUM\tCD\t_\t5\tpobj\t_\t_\n7\ta.m.\ta.m.\tNOUN\tNN\t_\t6\tadvmod\t_\t_\n8\tET\tET\tPROPN\tNNP\t_\t1\tappos\t_\t_\n9\t:\t:\tPUNCT\t:\t_\t13\tpunct\t_\t_\n10\tChris\tChris\tPROPN\tNNP\t_\t11\tcompound\t_\t_\n11\tMeekins\tMeekins\tPROPN\tNNP\t_\t13\tnsubj\t_\t_\n12\twill\twill\tAUX\tMD\t_\t13\taux\t_\t_\n13\tbe\tbe\tAUX\tVB\t_\t0\tROOT\t_\t_\n14\ton\ton\tADP\tRP\t_\t13\tprep\t_\t_\n15\tto\tto\tPART\tTO\t_\t16\taux\t_\t_\n16\tdiscuss\tdiscuss\tVERB\tVB\t_\t13\txcomp\t_\t_\n17\tthe\tthe\tDET\tDT\t_\t19\tdet\t_\t_\n18\tlatest\tlate\tADJ\tJJS\t_\t19\tamod\t_\t_\n19\tupdates\tupdate\tNOUN\tNNS\t_\t16\tdobj\t_\t_\n20\ton\ton\tADP\tIN\t_\t19\tprep\t_\t_\n21\tthe\tthe\tDET\tDT\t_\t23\tdet\t_\t_\n22\t#\t#\tSYM\t$\t_\t23\tcompound\t_\t_\n23\tcoronavirus\tcoronavirus\tNOUN\tNN\t_\t20\tpobj\t_\t_\n24\t.\t.\tPUNCT\t.\t_\t13\tpunct\t_\t_"
   },
   "China has more cases of coronavirus than it had of SARS, but a vaccine could be on the horizon #Topbuzz": {
      "aspects": {
         "coronavirus": 0,
         "sars": 1,
         "vaccine": 2
      },
      "aspect_sentiments_raw": {
         "0": "The sentiment towards coronavirus in the given sentence is neutral. Because the sentence provides a factual comparison between the number of coronavirus cases and SARS cases without expressing a specific positive or negative opinion about the virus itself, and instead ends with a hopeful note about a potential vaccine.",
         "1": "The sentiment towards SARS in the given sentence is neutral. Because the sentence uses SARS as a reference point to compare the number of cases with the current coronavirus situation without expressing any positive or negative feelings towards SARS itself.",
         "2": "The sentiment towards vaccine in the given sentence is positive. Because the phrase \"a vaccine could be on the horizon\" implies hope and anticipation, suggesting a positive outlook on the potential development and impact of a vaccine for the coronavirus."
      },
      "aspect_sentiments_label": {
         "0": "neutral",
         "1": "neutral",
         "2": "positive"
      },
      "aspect_syntactic": {
         "0": "Based on the provided CoNLL-U format, the word \"coronavirus\" (ID: 6) is a proper noun (NOUN/NN) and functions as the object of the preposition \"of\" (ID: 5). It is part of the phrase \"more cases of coronavirus,\" where \"coronavirus\" is the complement of the preposition \"of.\" This phrase is used to describe the subject \"cases\" (ID: 4), which is modified by \"more\" (ID: 3). The entire phrase \"more cases of coronavirus\" serves as the direct object (dobj) of the verb \"has\" (ID: 2).\n\nIn summary, \"coronavirus\" is used in the context of comparing the number of cases to those of SARS, indicating that there are more cases of coronavirus than there were of SARS in China.",
         "1": "In the given sentence, \"SARS\" is represented by the word at position 11 in the sentence. According to the CoNLL-U format provided:\n\n- **ID**: 11\n- **TEXT**: SARS\n- **LEMMA**: SARS\n- **POS**: PROPN (Proper Noun)\n- **TAG**: NNP (Proper Noun, Singular)\n- **FEATS**: _\n- **HEAD**: 9\n- **DEPREL**: pobj (prepositional object)\n- **DEPS**: _\n- **MISC**: _\n\nThis indicates that \"SARS\" functions as the prepositional object of the preposition \"of\" (which is at position 10). It is part of the phrase \"of SARS,\" which modifies the verb \"had\" (position 9). Essentially, \"SARS\" is used in a comparative context, indicating that China had fewer SARS cases compared to the current number",
         "2": "Based on the provided CoNLL-U format, the word \"vaccine\" (ID: 15) is a noun (NOUN) with the lemma \"vaccine\". It functions as the subject (nsubj) of the verb \"be\" (ID: 17). The phrase \"a vaccine could be on the horizon\" suggests a potential future development regarding a vaccine for the coronavirus.\n\nIn terms of syntactic dependencies:\n- \"a\" (ID: 14) is the determiner (det) of \"vaccine\".\n- \"could\" (ID: 16) is an auxiliary (aux) of \"be\".\n- \"be\" (ID: 17) is the main verb, and \"vaccine\" is its subject.\n- \"on\" (ID: 18) is a preposition (prep) related to \"be\", with \"horizon\" (ID: 20) as its object (pobj).\n\n"
      },
      "aspect_opinions": {
         "0": "The given sentence does not explicitly convey the speaker's personal opinion toward 'coronavirus.' Instead, it provides a factual comparison between the number of coronavirus cases and SARS cases in China, along with a hopeful note about a potential vaccine. The tone seems informative rather than expressing a specific sentiment or opinion about the virus itself.",
         "1": "The sentence does not explicitly convey the speaker's opinion toward SARS but rather provides a factual comparison between the number of coronavirus cases and past SARS cases in China. The mention of SARS is used to contextualize the scale of the current situation with the coronavirus. There's no direct sentiment expressed about SARS itself; it serves as a reference point for understanding the magnitude of the current health crisis.",
         "2": "The sentence does not explicitly state the speaker's opinion toward the vaccine but implies a cautiously optimistic outlook. The phrase \"a vaccine could be on the horizon\" suggests hope and anticipation for a potential solution to the coronavirus situation, indicating a positive view of the vaccine's possible development and impact."
      },
      "aspect_emotions_raw": {
         "0": "Emotion: neutral\n\nReasoning: The sentence does not express a clear emotional stance towards 'coronavirus.' It presents a factual comparison regarding the number of cases and mentions the possibility of a vaccine without conveying a strong positive or negative sentiment. Therefore, the most appropriate label for the emotion expressed towards 'coronavirus' in this context is 'neutral.'",
         "1": "Emotion: neutral\n\nReasoning: The sentence does not express any explicit emotion towards SARS. It merely uses SARS as a reference point to compare the scale of the current coronavirus cases in China. Since there is no indication of feelings such as fear, sadness, or any other emotional response towards SARS, the appropriate label is \"neutral.\"",
         "2": "Emotion: hopeful\n\nReasoning: The sentence expresses a sense of anticipation and hope with the phrase \"a vaccine could be on the horizon.\" This indicates that the speaker is looking forward to a potential resolution to the current situation involving the coronavirus, which aligns most closely with the emotion of hopefulness. There is an underlying optimism but the primary emotion conveyed is hope for the future development of a vaccine."
      },
      "aspect_emotions_label": {
         "0": "neutral",
         "1": "neutral",
         "2": "hopeful"
      },
      "conllu_parse": "# text = China has more cases of coronavirus than it had of SARS, but a vaccine could be on the horizon #Topbuzz\n1\tChina\tChina\tPROPN\tNNP\t_\t2\tnsubj\t_\t_\n2\thas\thave\tVERB\tVBZ\t_\t0\tROOT\t_\t_\n3\tmore\tmore\tADJ\tJJR\t_\t4\tamod\t_\t_\n4\tcases\tcase\tNOUN\tNNS\t_\t2\tdobj\t_\t_\n5\tof\tof\tADP\tIN\t_\t4\tprep\t_\t_\n6\tcoronavirus\tcoronavirus\tNOUN\tNN\t_\t5\tpobj\t_\t_\n7\tthan\tthan\tSCONJ\tIN\t_\t9\tmark\t_\t_\n8\tit\tit\tPRON\tPRP\t_\t9\tnsubj\t_\t_\n9\thad\thave\tVERB\tVBD\t_\t2\tadvcl\t_\t_\n10\tof\tof\tADP\tIN\t_\t9\tprep\t_\t_\n11\tSARS\tSARS\tPROPN\tNNP\t_\t10\tpobj\t_\t_\n12\t,\t,\tPUNCT\t,\t_\t2\tpunct\t_\t_\n13\tbut\tbut\tCCONJ\tCC\t_\t2\tcc\t_\t_\n14\ta\ta\tDET\tDT\t_\t15\tdet\t_\t_\n15\tvaccine\tvaccine\tNOUN\tNN\t_\t17\tnsubj\t_\t_\n16\tcould\tcould\tAUX\tMD\t_\t17\taux\t_\t_\n17\tbe\tbe\tAUX\tVB\t_\t2\tconj\t_\t_\n18\ton\ton\tADP\tIN\t_\t17\tprep\t_\t_\n19\tthe\tthe\tDET\tDT\t_\t20\tdet\t_\t_\n20\thorizon\thorizon\tNOUN\tNN\t_\t18\tpobj\t_\t_\n21\t#\t#\tSYM\t$\t_\t17\tpunct\t_\t_\n22\tTopbuzz\tTopbuzz\tPROPN\tNNP\t_\t17\tattr\t_\t_"
   }
}

### Teacher model aspect extraction

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct"
QUANTS = 4
MODEL_DIR = "/home/jovyan/models/Qwen2.5-32B-Instruct"

In [ ]:
model, tokenizer = load_model(
    MODEL_NAME, QUANTS, device_map="auto", cache_dir=MODEL_DIR
)

In [ ]:
tweets = example_data.keys()
conllus = [v["conllu_parse"] for k,v in example_data.items()]
prompts_r1 = prompt_aspect_extraction(tweets, conllus)

In [ ]:
texts = [
    tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    for messages in prompts_r1
]

In [ ]:
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=1024,
    return_tensors="pt",
).to(model.device)

In [ ]:
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )

In [ ]:
all_decoded = []
input_len = inputs.input_ids.shape[1]
generated_ids = outputs[:, input_len:]

decoded = tokenizer.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

all_decoded.extend(decoded)

In [ ]:
aspect_lists = [extract_aspects(x) for x in decoded]

In [ ]:
aspect_lists

### Original model aspect extraction

In [ ]:
STUDENT_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MODEL_DIR = "/home/jovyan/models/Qwen2.5-7B-Instruct"
QUANTS = None

In [ ]:
student_model, student_tokenizer = load_model(
    STUDENT_MODEL_NAME, QUANTS, device_map="auto", cache_dir=MODEL_DIR
)

In [ ]:
tweets = example_data.keys()
conllus = [v["conllu_parse"] for k,v in example_data.items()]
prompts_r1 = prompt_aspect_extraction_no_reason(tweets, conllus)

In [ ]:
orig_student_texts = [
    student_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    for messages in prompts_r1
]

In [ ]:
orig_student_inputs = student_tokenizer(
    orig_student_texts,
    padding=True,
    truncation=True,
    max_length=1024,
    return_tensors="pt",
).to(model.device)

In [ ]:
with torch.no_grad():
    orig_student_outputs = student_model.generate(
        **orig_student_inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )

In [ ]:
student_decoded = []

In [ ]:
input_len = orig_student_inputs.input_ids.shape[1]
generated_ids = orig_student_outputs[:, input_len:]

decoded = student_tokenizer.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

student_decoded.extend(decoded)

In [ ]:
student_aspect_lists = [extract_aspects(x) for x in student_decoded]

In [ ]:
student_aspect_lists

In [ ]:
aspect_lists

### Trained model aspect extraction

In [ ]:
BASE_MODEL_PATH = "/home/jovyan/models/Meta-Llama-3-8B-Instruct"
CHECKPOINT_PATH = "/home/jovyan/synchain-absa-emotion/trained_models/checkpoint-570"
MAX_NEW_TOKENS = 256
example_data = {
    "The more you know \uf8ff\u00fc\u00e5\u00e0 #coronavirusec #coronavirus #washyourhands": {
        "conllu_parse": "# text = The more you know \uf8ff\u00fc\u00e5\u00e0 #coronavirusec #coronavirus #washyourhands\n1\tThe\tthe\tPRON\tDT\t_\t2\tadvmod\t_\t_\n2\tmore\tmore\tADV\tRBR\t_\t4\tadvmod\t_\t_\n3\tyou\tyou\tPRON\tPRP\t_\t4\tnsubj\t_\t_\n4\tknow\tknow\tVERB\tVBP\t_\t11\tparataxis\t_\t_\n5\t\uf8ff\u00fc\u00e5\u00e0\t\uf8ff\u00fc\u00e5\u00e0\tNOUN\tNN\t_\t7\tnmod\t_\t_\n6\t#\t#\tSYM\t$\t_\t7\tpunct\t_\t_\n7\tcoronavirusec\tcoronavirusec\tNOUN\tNN\t_\t4\tdobj\t_\t_\n8\t#\t#\tSYM\t$\t_\t9\tcompound\t_\t_\n9\tcoronavirus\tcoronavirus\tPROPN\tNNP\t_\t10\tcompound\t_\t_\n10\t#\t#\tSYM\t$\t_\t4\tdep\t_\t_\n11\twashyourhands\twashyourhand\tNOUN\tNNS\t_\t0\tROOT\t_\t_"
    },
    "Tune in to @CNBCWEX around 5:30 a.m. ET: Chris Meekins will be on to discuss the latest updates on the #coronavirus.": {
        "conllu_parse": "# text = Tune in to @CNBCWEX around 5:30 a.m. ET: Chris Meekins will be on to discuss the latest updates on the #coronavirus.\n1\tTune\ttune\tNOUN\tNN\t_\t13\tadvcl\t_\t_\n2\tin\tin\tADP\tIN\t_\t1\tadvmod\t_\t_\n3\tto\tto\tADP\tIN\t_\t1\tprep\t_\t_\n4\t@CNBCWEX\t@CNBCWEX\tPROPN\tNNP\t_\t3\tpobj\t_\t_\n5\taround\taround\tADP\tIN\t_\t1\tprep\t_\t_\n6\t5:30\t5:30\tNUM\tCD\t_\t5\tpobj\t_\t_\n7\ta.m.\ta.m.\tNOUN\tNN\t_\t6\tadvmod\t_\t_\n8\tET\tET\tPROPN\tNNP\t_\t1\tappos\t_\t_\n9\t:\t:\tPUNCT\t:\t_\t13\tpunct\t_\t_\n10\tChris\tChris\tPROPN\tNNP\t_\t11\tcompound\t_\t_\n11\tMeekins\tMeekins\tPROPN\tNNP\t_\t13\tnsubj\t_\t_\n12\twill\twill\tAUX\tMD\t_\t13\taux\t_\t_\n13\tbe\tbe\tAUX\tVB\t_\t0\tROOT\t_\t_\n14\ton\ton\tADP\tRP\t_\t13\tprep\t_\t_\n15\tto\tto\tPART\tTO\t_\t16\taux\t_\t_\n16\tdiscuss\tdiscuss\tVERB\tVB\t_\t13\txcomp\t_\t_\n17\tthe\tthe\tDET\tDT\t_\t19\tdet\t_\t_\n18\tlatest\tlate\tADJ\tJJS\t_\t19\tamod\t_\t_\n19\tupdates\tupdate\tNOUN\tNNS\t_\t16\tdobj\t_\t_\n20\ton\ton\tADP\tIN\t_\t19\tprep\t_\t_\n21\tthe\tthe\tDET\tDT\t_\t23\tdet\t_\t_\n22\t#\t#\tSYM\t$\t_\t23\tcompound\t_\t_\n23\tcoronavirus\tcoronavirus\tNOUN\tNN\t_\t20\tpobj\t_\t_\n24\t.\t.\tPUNCT\t.\t_\t13\tpunct\t_\t_"
    },
    "China has more cases of coronavirus than it had of SARS, but a vaccine could be on the horizon #Topbuzz": {
        "conllu_parse": "# text = China has more cases of coronavirus than it had of SARS, but a vaccine could be on the horizon #Topbuzz\n1\tChina\tChina\tPROPN\tNNP\t_\t2\tnsubj\t_\t_\n2\thas\thave\tVERB\tVBZ\t_\t0\tROOT\t_\t_\n3\tmore\tmore\tADJ\tJJR\t_\t4\tamod\t_\t_\n4\tcases\tcase\tNOUN\tNNS\t_\t2\tdobj\t_\t_\n5\tof\tof\tADP\tIN\t_\t4\tprep\t_\t_\n6\tcoronavirus\tcoronavirus\tNOUN\tNN\t_\t5\tpobj\t_\t_\n7\tthan\tthan\tSCONJ\tIN\t_\t9\tmark\t_\t_\n8\tit\tit\tPRON\tPRP\t_\t9\tnsubj\t_\t_\n9\thad\thave\tVERB\tVBD\t_\t2\tadvcl\t_\t_\n10\tof\tof\tADP\tIN\t_\t9\tprep\t_\t_\n11\tSARS\tSARS\tPROPN\tNNP\t_\t10\tpobj\t_\t_\n12\t,\t,\tPUNCT\t,\t_\t2\tpunct\t_\t_\n13\tbut\tbut\tCCONJ\tCC\t_\t2\tcc\t_\t_\n14\ta\ta\tDET\tDT\t_\t15\tdet\t_\t_\n15\tvaccine\tvaccine\tNOUN\tNN\t_\t17\tnsubj\t_\t_\n16\tcould\tcould\tAUX\tMD\t_\t17\taux\t_\t_\n17\tbe\tbe\tAUX\tVB\t_\t2\tconj\t_\t_\n18\ton\ton\tADP\tIN\t_\t17\tprep\t_\t_\n19\tthe\tthe\tDET\tDT\t_\t20\tdet\t_\t_\n20\thorizon\thorizon\tNOUN\tNN\t_\t18\tpobj\t_\t_\n21\t#\t#\tSYM\t$\t_\t17\tpunct\t_\t_\n22\tTopbuzz\tTopbuzz\tPROPN\tNNP\t_\t17\tattr\t_\t_"
    }
}


In [ ]:
llama_model, llama_tokenizer = load_trained_model(BASE_MODEL_PATH, CHECKPOINT_PATH)

In [ ]:
aspects, raw = run_aspect_extraction(llama_model, llama_tokenizer, text, conllu_parse, max_new_tokens=128)
print(aspects)